### Import Section
*Importing the neccessary packages and libraries.*

In [2]:
import csv
import os
from pathlib import Path

import pyodbc
from dotenv import load_dotenv

In [3]:
# Building the tables list and it will be list of dictionaries contains the full info about each table.
TABLES = [
    {
        "name": "crm_cust_info",
        "file": "datasets/source_crm/cust_info.csv",
        "columns": [
            "cst_id",
            "cst_key",
            "cst_firstname",
            "cst_lastname",
            "cst_marital_status",
            "cst_gndr",
            "cst_create_date",
        ],
        "keys": ["cst_id"],
        "order_by": "cst_create_date DESC",
    },
    {
        "name": "crm_prd_info",
        "file": "datasets/source_crm/prd_info.csv",
        "columns": [
            "prd_id",
            "prd_key",
            "prd_nm",
            "prd_cost",
            "prd_line",
            "prd_start_dt",
            "prd_end_dt",
        ],
        "keys": ["prd_id"],
    },
    {
        "name": "crm_sales_details",
        "file": "datasets/source_crm/sales_details.csv",
        "columns": [
            "sls_ord_num",
            "sls_prd_key",
            "sls_cust_id",
            "sls_order_dt",
            "sls_ship_dt",
            "sls_due_dt",
            "sls_sales",
            "sls_quantity",
            "sls_price",
        ],
        "keys": ["sls_ord_num", "sls_prd_key"],
    },
    {
        "name": "erp_cust_az12",
        "file": "datasets/source_erp/CUST_AZ12.csv",
        "columns": ["CID", "BDATE", "GEN"],
        "keys": ["CID"],
    },
    {
        "name": "erp_loc_a101",
        "file": "datasets/source_erp/LOC_A101.csv",
        "columns": ["CID", "CNTRY"],
        "keys": ["CID"],
    },
    {
        "name": "erp_px_cat_g1v2",
        "file": "datasets/source_erp/PX_CAT_G1V2.csv",
        "columns": ["ID", "CAT", "SUBCAT", "MAINTENANCE"],
        "keys": ["ID"],
    },
]

In [7]:
# This is the function to get the connection
# to the SQL Server database using pyodbc and environment variables for configuration.
def get_connection():
    load_dotenv()

    driver = os.getenv("SQLSERVER_DRIVER", "ODBC Driver 18 for SQL Server")
    host = os.getenv("SQLSERVER_HOST", "localhost")
    port = os.getenv("SQLSERVER_PORT", "1433")
    database = os.getenv("SQLSERVER_DATABASE", "CustomerSales")
    user = os.getenv("SQLSERVER_USER", "sa")
    password = os.getenv("SQLSERVER_PASSWORD")
    encrypt = os.getenv("SQLSERVER_ENCRYPT", "yes")
    trust_certificate = os.getenv("SQLSERVER_TRUST_CERTIFICATE", "yes")

    # Validate that the password is provided
    if not password:
        raise RuntimeError("SQLSERVER_PASSWORD is missing from your .env file.")

    connection_string = (
        f"DRIVER={driver};"
        f"SERVER={host},{port};"
        f"DATABASE={database};"
        f"UID={user};"
        f"PWD={password};"
        f"Encrypt={encrypt};"
        f"TrustServerCertificate={trust_certificate};"
        "Connection Timeout=30;"
    )

    return pyodbc.connect(connection_string)